# `03_memory.ipynb`

## Key Concept
- State에 `messages` 항목에 대한 설명 -> Graph 내부에서 턴마다 필요한 메세지(AI, Human, System, Tool)를 쌓는 용도
- `InMemorySaver`로 테스트 -> 여러 턴에 대해 저장하자
- `Postgres` 에 직접 저장하는 방법

In [12]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
# 직접 만들기 (교육용)
from typing import TypedDict, Annotated
from langgraph.graph import add_messages

class MyState(TypedDict):
    # messages: list  # 그냥 리스트임 -> 교체해야하면 교체됨
    messages: Annotated[list, add_messages]  # 교체할 타이밍에, 교체하지 않고 쌓아 나가는 기능을 추가해주세요
    is_good: bool

In [14]:
# 앞으로 모든 state는 이렇게 만든다.
from langgraph.graph import MessagesState

class MyState(MessagesState):
    # messages 기능 자동 탑재
    is_good: bool

In [15]:
# node
from langchain.chat_models import init_chat_model

llm = init_chat_model('openai:gpt-4.1-mini')

# is_good 처리
def node_a(state: MyState):
    return {'is_good': True}  # 기존 state의 'is_good' 을 바꿔주세요


# AI 답변 생성
def node_b(state: MyState):
    messages = state['messages']
    ai_msg = llm.invoke(messages)
    return {'messages': [ai_msg]}  # 기존 state의 'messages'를 교체해 주세요

In [16]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

graph = StateGraph(MyState)
graph.add_node(node_a)
graph.add_node(node_b)
graph.add_edge(START, 'node_a')
graph.add_edge('node_a', 'node_b')
graph.add_edge('node_b', END)


In [17]:
from langchain.messages import HumanMessage
# 테스트용 메모리
workflow = graph.compile(checkpointer=InMemorySaver())
config = {'configurable': {'thread_id': '123-456'}}  # 각 세션의 고유 id로 구분

result = workflow.invoke(
    {'messages': [HumanMessage('굿굿')]},  # 1번 인자: state
    config,  # 2번 인자, 설정값
)

In [18]:
for msg in result['messages']:
    msg.pretty_print()

================================ Human Message =================================

굿굿
================================== Ai Message ==================================

감사합니다! 도움이 필요하시면 언제든 말씀해 주세요. 😊


## Postgres 영구저장

In [ ]:
# 영구저장 메모리
# uv add langgraph-checkpoint-postgres psycopg[binary]
import os
from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver

